In [ ]:
import json
import time
import os
from typing import Dict, Any, List, Optional, Iterable, Tuple, Union, Set, Callable
from pathlib import Path

import pandas as pd
import requests

# ============================================================
# ZWEITE PIPELINE: THREAD-LEVEL-METRIKEN
#  - argument_novelty  (in [0,1])
#  - semantic_entropy  (>= 0, nicht nach oben begrenzt)
#  Input: NDJSON der ersten Pipeline (mit arguments, parent_id)
# ============================================================

# ------------------------
# Task-Spezifikationen (Pipeline 2)
# ------------------------

THREAD_TASK_SPECS: Dict[str, Dict[str, Any]] = {
    "argument_novelty": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "argument_novelty",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),          # 0 = kein neues Argument, 1 = komplett neu
        "allow_abstain": True,
        "label_key": None,
    },
    "semantic_entropy": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "semantic_entropy",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, float("inf")), 
        "allow_abstain": True,
        "label_key": None,
    },
}

# ------------------------
# System Prompt für Pipeline 2
# ------------------------

THREAD_SYSTEM_PROMPT = """You are an annotator estimating discourse-level metrics for a given comment
in the context of all previous comments in the discussion thread.

You receive:
{
  "CURRENT_ARGUMENTS": [ ... ],   // arguments extracted for the current comment
  "HISTORY_ARGUMENTS": [ ... ]    // arguments from ALL earlier comments in the thread,
                                  // following the parent_id chain back to the root
}

HISTORY_ARGUMENTS is ordered from earliest ancestor to the immediate parent.

Your tasks:

1) Argument novelty (argument_novelty)
--------------------------------------
TASK: Estimate how much of the CURRENT_ARGUMENTS content is *novel* relative to HISTORY_ARGUMENTS.

Score: float in [0,1].
- 0.0 = entirely re-uses arguments already present in HISTORY_ARGUMENTS
- 0.5 = mix of re-used and somewhat new arguments
- 1.0 = introduces entirely new arguments not present before

Heuristics:
- If HISTORY_ARGUMENTS is empty (no previous comments), default to a high score (e.g. 0.8-1.0),
  unless CURRENT_ARGUMENTS itself is empty or trivial.
- If CURRENT_ARGUMENTS is empty, return "ABSTAIN" (you cannot judge novelty).

JSON schema:
{
  "task": "argument_novelty",
  "score": <float in [0.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

2) Semantic entropy (semantic_entropy)
--------------------------------------
TASK: Approximate the semantic diversity of arguments in the thread up to and including the current comment.

Intuition: a Shannon-entropy-like measure of the argument-topic distribution.
- Low entropy (~0): almost all arguments are about one narrow theme.
- Medium entropy (~1-2): several recurring argument themes.
- High entropy (>2): many different themes with relatively balanced presence.

Score: non-negative float (>= 0), no strict upper bound, but usual values should stay in a
reasonable range (e.g., 0 to 3).

Heuristics:
- Base your judgement on how many distinct argument themes you see in
  HISTORY_ARGUMENTS + CURRENT_ARGUMENTS, and how balanced they are.
- If there are no arguments at all, or almost no content, return "ABSTAIN".

JSON schema:
{
  "task": "semantic_entropy",
  "score": <float >= 0.0 OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

General rules:
--------------
- Prefer "ABSTAIN" if information is too weak, arguments are empty, or you cannot judge reliably.
- Output MUST be strictly valid JSON with the exact schema above.
"""

THREAD_TASK_INSTRUCTION_TEMPLATE = (
    "Now perform ONLY the task = {task_name} on the given CURRENT_ARGUMENTS and HISTORY_ARGUMENTS. "
    "Return strictly valid JSON for that task and nothing else (no Markdown). "
    "Ensure keys and value ranges match the schema exactly."
)

# ------------------------
# Validation für Pipeline 2
# ------------------------

def validate_thread_response(task: str, obj: Dict[str, Any]) -> Optional[str]:
    """Validate response for THREAD_TASK_SPECS (argument_novelty, semantic_entropy)."""
    spec = THREAD_TASK_SPECS[task]

    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    missing = spec["schema_keys"] - set(obj.keys())
    if missing:
        return f"Missing required keys: {sorted(missing)}"

    if obj.get("task") != spec["task_value"]:
        return f'Field "task" must be "{spec["task_value"]}"'

    conf = obj.get("confidence")
    if not _is_number(conf):
        return '"confidence" must be a number'
    lo_c, hi_c = spec["confidence_range"]
    if not (lo_c <= conf <= hi_c):
        return f'"confidence" must be in [{lo_c}, {hi_c}]'

    score_key = spec["score_key"]
    val = obj.get(score_key)

    # ABSTAIN erlaubt?
    if isinstance(val, str):
        if spec.get("allow_abstain") and val == "ABSTAIN":
            return None
        return f'"{score_key}" must be a number or "ABSTAIN"'

    if not _is_number(val):
        return f'"{score_key}" must be a number'

    lo, hi = spec["score_range"]
    if not (lo <= float(val) <= hi):
        return f'"{score_key}" out of range [{lo}, {hi}]'

    return None


# ------------------------
# Prompt für Pipeline 2 bauen
# ------------------------

def build_thread_user_prompt(
    task: str,
    current_arguments: List[str],
    history_arguments: List[str],
) -> str:
    payload = {
        "CURRENT_ARGUMENTS": current_arguments,
        "HISTORY_ARGUMENTS": history_arguments,
    }
    directive = THREAD_TASK_INSTRUCTION_TEMPLATE.format(task_name=task)
    return json.dumps(payload, ensure_ascii=False) + "\n\n" + directive


# ------------------------
# Ein Task-Aufruf (Pipeline 2)
# ------------------------

def annotate_thread_metric(
    task: str,
    current_arguments: List[str],
    history_arguments: List[str],
) -> Dict[str, Any]:
    """
    Run one thread-level task (argument_novelty or semantic_entropy) for a given comment,
    given its CURRENT_ARGUMENTS and HISTORY_ARGUMENTS (from ancestor comments).
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": THREAD_SYSTEM_PROMPT},
                {"role": "user", "content": build_thread_user_prompt(task, current_arguments, history_arguments)},
            ]
            raw = call_lmstudio_chat(messages, temperature=0.0)
            obj = json.loads(raw)

            err = validate_thread_response(task, obj)
            if err is None:
                return obj
            else:
                last_err = f"Thread schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            last_err = f"HTTP error (attempt {attempt}): {e}"
        except json.JSONDecodeError as e:
            last_err = f"JSON parse error (attempt {attempt}): {e}"

        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    return {
        "task": task,
        "error": last_err or "Unknown error",
    }


# ------------------------
# Helper: Kommentare + Parent-Kette aus NDJSON laden
# ------------------------

def load_comments_from_labels_ndjson(
    ndjson_path: str,
) -> Dict[str, Dict[str, Any]]:
    """
    Liest die NDJSON-Ausgabe der ersten Pipeline ein und extrahiert
    pro comment_id genau einen Eintrag mit:
        - comment_id
        - comment_index
        - parent_id
        - arguments
        - arguments_confidence

    Annahme: für eine gegebene comment_id sind arguments in allen Tasks identisch,
    daher reicht es, die erste Zeile mit dieser id zu nehmen.
    """
    mapping: Dict[str, Dict[str, Any]] = {}

    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            cid = str(obj.get("comment_id", ""))
            if not cid:
                continue
            if cid in mapping:
                # schon gesehen -> überspringen
                continue

            mapping[cid] = {
                "comment_id": cid,
                "comment_index": obj.get("comment_index", None),
                "parent_id": obj.get("parent_id", None),
                "arguments": obj.get("arguments", []),
                "arguments_confidence": obj.get("arguments_confidence", None),
            }

    return mapping


def build_history_arguments_for_comment(
    comment_id: str,
    comment_map: Dict[str, Dict[str, Any]],
) -> List[str]:
    """
    Baut HISTORY_ARGUMENTS für einen Kommentar auf, indem iterativ über parent_id
    bis zum Thread-Anfang gelaufen wird.

    Reihenfolge: frühester Vorfahre zuerst, unmittelbarer Parent zuletzt.
    """
    history: List[Tuple[str, List[str]]] = []
    seen: Set[str] = set()

    current_parent = comment_map.get(comment_id, {}).get("parent_id")

    # Parent-Kette nach oben wandern
    while current_parent is not None and current_parent in comment_map and current_parent not in seen:
        seen.add(current_parent)
        parent_entry = comment_map[current_parent]
        parent_args = parent_entry.get("arguments", [])
        history.append((current_parent, parent_args))
        current_parent = parent_entry.get("parent_id")

    # jetzt von Wurzel zum direkten Parent sortieren
    history.reverse()

    # nur Argument-Strings flach zurückgeben
    history_arguments: List[str] = []
    for cid, args in history:
        for a in args:
            history_arguments.append(f"[{cid}] {a}")

    return history_arguments


# ------------------------
# Haupt-Pipeline 2: Thread-Metriken
# ------------------------

def run_thread_metrics_pipeline(
    ndjson_in: str,
    ndjson_out: str = "thread_metrics.ndjson",
    tasks: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """
    Zweite Pipeline:
    - liest NDJSON von Pipeline 1 (per-comment-Tasks mit arguments)
    - rekonstruiert für jede comment_id die parent_id-Kette
    - ruft für jede comment_id und jeden Task (argument_novelty, semantic_entropy)
      LM Studio auf
    - schreibt NDJSON mit einem Eintrag pro (comment_id, task)

    Output-Schema pro Zeile:
    {
      "comment_id": "...",
      "comment_index": <int or null>,
      "parent_id": "... or null",
      "task": "argument_novelty" | "semantic_entropy",
      "result": { ...thread-task-json... },
      "current_arguments": [...],
      "history_arguments": [...]
    }
    """
    if tasks is None:
        tasks = list(THREAD_TASK_SPECS.keys())

    # Kommentare + Parent-Infos + Arguments aus erster Pipeline laden
    comment_map = load_comments_from_labels_ndjson(ndjson_in)
    comment_ids = list(comment_map.keys())

    written_records = 0
    error_count = 0

    with open(ndjson_out, "w", encoding="utf-8") as fp:
        for cid in comment_ids:
            entry = comment_map[cid]
            current_args = entry.get("arguments", [])
            parent_id = entry.get("parent_id", None)
            comment_index = entry.get("comment_index", None)

            # HISTORY_ARGUMENTS aus Parent-Kette
            history_args = build_history_arguments_for_comment(cid, comment_map)

            for task in tasks:
                try:
                    result = annotate_thread_metric(
                        task=task,
                        current_arguments=current_args,
                        history_arguments=history_args,
                    )
                    out = {
                        "comment_id": cid,
                        "comment_index": comment_index,
                        "parent_id": parent_id,
                        "task": task,
                        "result": result,
                        "current_arguments": current_args,
                        "history_arguments": history_args,
                    }
                    write_ndjson_line(fp, out)
                    written_records += 1
                except Exception:
                    error_count += 1

    return {
        "num_comments": len(comment_ids),
        "written_records": written_records,
        "errors": error_count,
    }
